# Phân Loại Ảnh Sử Dụng SVM (Support Vector Machine)

Notebook này thực hiện:
- Tải và tiền xử lý dataset CIFAR-10
- Trích xuất đặc trưng HOG (Histogram of Oriented Gradients)
- Huấn luyện mô hình SVM
- Đánh giá mô hình với các metrics: Accuracy, Precision, Recall, F1-score, IoU (Jaccard), Confusion Matrix, AUC-ROC

## 1. Cài đặt thư viện

In [ ]:
# TensorFlow đã được cài sẵn trong Google Colab.
# Chạy lệnh dưới để đảm bảo các thư viện khác đúng phiên bản.
!pip install "scikit-image>=0.21" "scikit-learn>=1.3" "matplotlib>=3.7" "numpy>=1.24" "seaborn>=0.12" "joblib>=1.3" -q

## 2. Import thư viện

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dataset
from tensorflow.keras.datasets import cifar10

# Feature extraction
from skimage.feature import hog
from skimage.color import rgb2gray

# SVM và preprocessing
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Metrics đánh giá
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,          # IoU score
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc
)

# Lưu mô hình
import joblib
import time

print('Tất cả thư viện đã được import thành công!')

## 3. Tải và Khám Phá Dataset (CIFAR-10)

CIFAR-10 gồm 60,000 ảnh màu 32x32 pixels thuộc 10 lớp:
0: Airplane, 1: Automobile, 2: Bird, 3: Cat, 4: Deer,
5: Dog, 6: Frog, 7: Horse, 8: Ship, 9: Truck

In [ ]:
# Tải CIFAR-10
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

# Tên các lớp
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# Flatten nhãn
y_train_raw = y_train_raw.flatten()
y_test_raw  = y_test_raw.flatten()

print(f'Kích thước tập train: {X_train_raw.shape}')
print(f'Kích thước tập test:  {X_test_raw.shape}')
print(f'Số lớp: {len(class_names)}')
print(f'Tên các lớp: {class_names}')

In [ ]:
# Hiển thị mẫu ảnh từ mỗi lớp
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Mẫu ảnh từ mỗi lớp trong CIFAR-10', fontsize=14, fontweight='bold')

for cls_idx in range(10):
    idx = np.where(y_train_raw == cls_idx)[0][0]
    ax = axes[cls_idx // 5][cls_idx % 5]
    ax.imshow(X_train_raw[idx])
    ax.set_title(class_names[cls_idx], fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('cifar10_samples.png', dpi=100, bbox_inches='tight')
plt.show()

# Phân phối lớp
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, y, title in zip(axes, [y_train_raw, y_test_raw], ['Train', 'Test']):
    unique, counts = np.unique(y, return_counts=True)
    ax.bar([class_names[i] for i in unique], counts, color='steelblue', edgecolor='black')
    ax.set_title(f'Phân phối lớp - {title} set', fontsize=12)
    ax.set_xlabel('Lớp')
    ax.set_ylabel('Số lượng mẫu')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Lấy Mẫu Con (Subset) để Tăng Tốc Độ Huấn Luyện

Do CIFAR-10 có 60,000 ảnh, việc dùng toàn bộ với SVM sẽ rất chậm.
Ta lấy subset cân bằng cho mỗi lớp.

In [ ]:
# Số mẫu mỗi lớp (tăng nếu muốn kết quả tốt hơn, nhưng sẽ chậm hơn)
N_TRAIN_PER_CLASS = 500   # 500 * 10 = 5,000 ảnh train
N_TEST_PER_CLASS  = 100   # 100 * 10 = 1,000 ảnh test

def balanced_subset(X, y, n_per_class, seed=42):
    np.random.seed(seed)
    indices = []
    for cls in range(10):
        cls_indices = np.where(y == cls)[0]
        chosen = np.random.choice(cls_indices, size=n_per_class, replace=False)
        indices.extend(chosen)
    indices = np.array(indices)
    np.random.shuffle(indices)
    return X[indices], y[indices]

X_train_sub, y_train_sub = balanced_subset(X_train_raw, y_train_raw, N_TRAIN_PER_CLASS)
X_test_sub,  y_test_sub  = balanced_subset(X_test_raw,  y_test_raw,  N_TEST_PER_CLASS)

print(f'Subset train: {X_train_sub.shape}, labels: {y_train_sub.shape}')
print(f'Subset test:  {X_test_sub.shape},  labels: {y_test_sub.shape}')

## 5. Trích Xuất Đặc Trưng HOG

HOG (Histogram of Oriented Gradients) là kỹ thuật mô tả hình dạng và cạnh của đối tượng trong ảnh.

In [ ]:
def extract_hog_features(images, orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2)):
    """Trích xuất đặc trưng HOG từ tập ảnh."""
    features = []
    for img in images:
        gray = rgb2gray(img)
        feat = hog(
            gray,
            orientations=orientations,
            pixels_per_cell=pixels_per_cell,
            cells_per_block=cells_per_block,
            feature_vector=True
        )
        features.append(feat)
    return np.array(features)

print('Đang trích xuất đặc trưng HOG từ tập train...')
start = time.time()
X_train_hog = extract_hog_features(X_train_sub)
print(f'  Hoàn thành! Thời gian: {time.time()-start:.1f}s | Shape: {X_train_hog.shape}')

print('Đang trích xuất đặc trưng HOG từ tập test...')
start = time.time()
X_test_hog  = extract_hog_features(X_test_sub)
print(f'  Hoàn thành! Thời gian: {time.time()-start:.1f}s | Shape: {X_test_hog.shape}')

In [ ]:
# Trực quan hóa HOG features
from skimage.feature import hog as hog_vis
from skimage import exposure

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Trực quan hóa HOG Features', fontsize=14, fontweight='bold')

for i in range(4):
    img = X_train_sub[i]
    gray = rgb2gray(img)
    _, hog_image = hog_vis(
        gray, orientations=9, pixels_per_cell=(4, 4),
        cells_per_block=(2, 2), visualize=True
    )
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

    axes[0][i].imshow(img)
    axes[0][i].set_title(f'Ảnh gốc: {class_names[y_train_sub[i]]}')
    axes[0][i].axis('off')

    axes[1][i].imshow(hog_image_rescaled, cmap='gray')
    axes[1][i].set_title('HOG Features')
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('hog_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Tiền Xử Lý Dữ Liệu

In [ ]:
# Chuẩn hóa đặc trưng
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_hog)
X_test_scaled  = scaler.transform(X_test_hog)

# Giảm chiều với PCA để tăng tốc độ
N_COMPONENTS = 100  # Giữ lại 100 thành phần chính
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

explained_var = np.sum(pca.explained_variance_ratio_) * 100
print(f'HOG features gốc: {X_train_hog.shape[1]} chiều')
print(f'Sau PCA ({N_COMPONENTS} thành phần): {X_train_pca.shape[1]} chiều')
print(f'Phương sai giải thích được: {explained_var:.1f}%')

# Vẽ biểu đồ phương sai tích lũy
plt.figure(figsize=(9, 4))
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
plt.plot(range(1, N_COMPONENTS + 1), cumvar, marker='o', markersize=3, color='steelblue')
plt.axhline(y=90, color='red', linestyle='--', label='90% phương sai')
plt.xlabel('Số thành phần PCA')
plt.ylabel('Phương sai tích lũy (%)')
plt.title('PCA - Phương sai tích lũy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Huấn Luyện Mô Hình SVM

Sử dụng `SVC` với kernel RBF và `probability=True` để tính AUC-ROC.

In [ ]:
print('Đang huấn luyện mô hình SVM (kernel=RBF)...')
start_time = time.time()

svm_model = SVC(
    kernel='rbf',          # Kernel RBF phù hợp với dữ liệu phi tuyến
    C=10.0,                # Tham số điều chỉnh margin
    gamma='scale',         # Gamma tự động scale
    probability=True,      # Cần thiết để tính AUC-ROC
    decision_function_shape='ovr',  # One-vs-Rest cho đa lớp
    random_state=42
)

svm_model.fit(X_train_pca, y_train_sub)

train_time = time.time() - start_time
print(f'Huấn luyện hoàn thành! Thời gian: {train_time:.1f}s')
print(f'Số support vectors: {sum(svm_model.n_support_)}')

## 8. Đánh Giá Mô Hình

### 8.1 Dự đoán

In [ ]:
# Dự đoán trên tập train và test
y_train_pred = svm_model.predict(X_train_pca)
y_test_pred  = svm_model.predict(X_test_pca)

# Xác suất dự đoán (cho AUC-ROC)
y_test_prob  = svm_model.predict_proba(X_test_pca)

print('Dự đoán hoàn thành!')
print(f'  Train predictions shape: {y_train_pred.shape}')
print(f'  Test predictions shape:  {y_test_pred.shape}')
print(f'  Test probabilities shape: {y_test_prob.shape}')

### 8.2 Các Metrics Đánh Giá

In [ ]:
# =========================================================
# TÍNH TOÁN CÁC METRICS
# =========================================================

# --- Accuracy ---
train_accuracy = accuracy_score(y_train_sub, y_train_pred)
test_accuracy  = accuracy_score(y_test_sub,  y_test_pred)

# --- Precision, Recall, F1-score (macro average) ---
precision = precision_score(y_test_sub, y_test_pred, average='macro')
recall    = recall_score   (y_test_sub, y_test_pred, average='macro')
f1        = f1_score       (y_test_sub, y_test_pred, average='macro')

# --- Precision, Recall, F1-score (weighted average) ---
precision_w = precision_score(y_test_sub, y_test_pred, average='weighted')
recall_w    = recall_score   (y_test_sub, y_test_pred, average='weighted')
f1_w        = f1_score       (y_test_sub, y_test_pred, average='weighted')

# --- IoU / Jaccard Score ---
# Jaccard Score = IoU = |A ∩ B| / |A ∪ B|
iou_macro    = jaccard_score(y_test_sub, y_test_pred, average='macro')
iou_weighted = jaccard_score(y_test_sub, y_test_pred, average='weighted')
iou_per_class = jaccard_score(y_test_sub, y_test_pred, average=None)

# --- AUC-ROC (One-vs-Rest, macro) ---
y_test_bin = label_binarize(y_test_sub, classes=range(10))
auc_roc = roc_auc_score(y_test_bin, y_test_prob, average='macro', multi_class='ovr')

# =========================================================
# IN KẾT QUẢ
# =========================================================
print('=' * 55)
print('       KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH SVM')
print('=' * 55)
print(f'  Accuracy (Train):          {train_accuracy*100:.2f}%')
print(f'  Accuracy (Test):           {test_accuracy*100:.2f}%')
print('-' * 55)
print(f'  Precision (macro):         {precision*100:.2f}%')
print(f'  Recall    (macro):         {recall*100:.2f}%')
print(f'  F1-score  (macro):         {f1*100:.2f}%')
print('-' * 55)
print(f'  Precision (weighted):      {precision_w*100:.2f}%')
print(f'  Recall    (weighted):      {recall_w*100:.2f}%')
print(f'  F1-score  (weighted):      {f1_w*100:.2f}%')
print('-' * 55)
print(f'  IoU / Jaccard (macro):     {iou_macro*100:.2f}%')
print(f'  IoU / Jaccard (weighted):  {iou_weighted*100:.2f}%')
print('-' * 55)
print(f'  AUC-ROC (macro, OvR):      {auc_roc:.4f}')
print('=' * 55)

### 8.3 Metrics theo từng lớp

In [ ]:
# Classification report chi tiết
report = classification_report(
    y_test_sub, y_test_pred,
    target_names=class_names,
    digits=4
)
print('Classification Report:')
print(report)

# IoU từng lớp
print('IoU (Jaccard Score) từng lớp:')
print('-' * 30)
for cls_name, iou_val in zip(class_names, iou_per_class):
    print(f'  {cls_name:<12}: {iou_val*100:.2f}%')

### 8.4 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_sub, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Confusion matrix - số lượng
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Số lượng)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Nhãn dự đoán', fontsize=11)
axes[0].set_ylabel('Nhãn thực tế', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Confusion matrix - tỷ lệ
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Greens',
    xticklabels=class_names, yticklabels=class_names,
    ax=axes[1]
)
axes[1].set_title('Confusion Matrix (Tỷ lệ)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nhãn dự đoán', fontsize=11)
axes[1].set_ylabel('Nhãn thực tế', fontsize=11)
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

### 8.5 AUC-ROC Curve

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, 10))

plt.figure(figsize=(10, 8))

for i, (cls_name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_prob[:, i])
    roc_auc_i   = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=1.5,
             label=f'{cls_name} (AUC = {roc_auc_i:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR / Recall)', fontsize=12)
plt.title(f'AUC-ROC Curve - SVM (Macro AUC = {auc_roc:.4f})', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=100, bbox_inches='tight')
plt.show()

### 8.6 Tổng Hợp Metrics theo từng lớp

In [ ]:
# Tính precision, recall, f1 từng lớp
precision_per_class = precision_score(y_test_sub, y_test_pred, average=None)
recall_per_class    = recall_score   (y_test_sub, y_test_pred, average=None)
f1_per_class        = f1_score       (y_test_sub, y_test_pred, average=None)

x = np.arange(len(class_names))
width = 0.2

fig, ax = plt.subplots(figsize=(15, 6))
bars1 = ax.bar(x - width*1.5, precision_per_class, width, label='Precision', color='steelblue', alpha=0.85)
bars2 = ax.bar(x - width*0.5, recall_per_class,    width, label='Recall',    color='seagreen',  alpha=0.85)
bars3 = ax.bar(x + width*0.5, f1_per_class,        width, label='F1-score',  color='darkorange', alpha=0.85)
bars4 = ax.bar(x + width*1.5, iou_per_class,       width, label='IoU',       color='orchid',     alpha=0.85)

ax.set_xlabel('Lớp', fontsize=12)
ax.set_ylabel('Giá trị', fontsize=12)
ax.set_title('Metrics đánh giá theo từng lớp (SVM)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_per_class.png', dpi=100, bbox_inches='tight')
plt.show()

### 8.7 Trực Quan Hóa Kết Quả Dự Đoán

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
fig.suptitle('Kết quả dự đoán của SVM\n(Xanh = đúng, Đỏ = sai)', fontsize=13, fontweight='bold')

np.random.seed(7)
sample_indices = np.random.choice(len(X_test_sub), size=18, replace=False)

for idx, ax in zip(sample_indices, axes.flatten()):
    ax.imshow(X_test_sub[idx])
    true_label = class_names[y_test_sub[idx]]
    pred_label = class_names[y_test_pred[idx]]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'T: {true_label}\nP: {pred_label}', fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Tổng Kết và Lưu Mô Hình

In [ ]:
print('=' * 60)
print('        TỔNG KẾT MÔ HÌNH SVM - CIFAR-10')
print('=' * 60)
print(f'  Dataset:           CIFAR-10')
print(f'  Train samples:     {len(y_train_sub):,} ({N_TRAIN_PER_CLASS}/lớp)')
print(f'  Test samples:      {len(y_test_sub):,} ({N_TEST_PER_CLASS}/lớp)')
print(f'  HOG features:      {X_train_hog.shape[1]} chiều')
print(f'  Sau PCA:           {N_COMPONENTS} chiều ({explained_var:.1f}% phương sai)')
print(f'  Kernel SVM:        RBF (C=10, gamma=scale)')
print(f'  Support vectors:   {sum(svm_model.n_support_):,}')
print(f'  Thời gian train:   {train_time:.1f}s')
print('-' * 60)
print('  METRICS TRÊN TẬP TEST:')
print(f'  Accuracy:          {test_accuracy*100:.2f}%')
print(f'  Precision (macro): {precision*100:.2f}%')
print(f'  Recall    (macro): {recall*100:.2f}%')
print(f'  F1-score  (macro): {f1*100:.2f}%')
print(f'  IoU       (macro): {iou_macro*100:.2f}%')
print(f'  AUC-ROC   (macro): {auc_roc:.4f}')
print('=' * 60)

In [ ]:
# Lưu mô hình và các thành phần tiền xử lý
joblib.dump(svm_model, 'svm_model.pkl')
joblib.dump(scaler,    'scaler.pkl')
joblib.dump(pca,       'pca.pkl')

print('Đã lưu mô hình:')
print('  - svm_model.pkl')
print('  - scaler.pkl')
print('  - pca.pkl')

## 10. Demo Dự Đoán Ảnh Mới

In [ ]:
def predict_single_image(image, model, scaler, pca):
    """Dự đoán nhãn cho một ảnh đầu vào."""
    gray = rgb2gray(image)
    feat = hog(gray, orientations=9, pixels_per_cell=(4, 4),
               cells_per_block=(2, 2), feature_vector=True)
    feat_scaled = scaler.transform(feat.reshape(1, -1))
    feat_pca    = pca.transform(feat_scaled)
    pred_label  = model.predict(feat_pca)[0]
    pred_proba  = model.predict_proba(feat_pca)[0]
    confidence  = pred_proba[pred_label]
    return pred_label, confidence, pred_proba

# Demo trên 6 ảnh ngẫu nhiên
np.random.seed(99)
demo_indices = np.random.choice(len(X_test_sub), size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Demo Dự Đoán Ảnh Đơn lẻ', fontsize=13, fontweight='bold')

for ax, idx in zip(axes.flatten(), demo_indices):
    img = X_test_sub[idx]
    true_cls = y_test_sub[idx]
    pred_cls, conf, proba = predict_single_image(img, svm_model, scaler, pca)

    ax.imshow(img)
    color = 'green' if pred_cls == true_cls else 'red'
    ax.set_title(
        f'Thực tế: {class_names[true_cls]}\n'
        f'Dự đoán: {class_names[pred_cls]} ({conf*100:.1f}%)',
        color=color, fontsize=9, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('demo_predictions.png', dpi=100, bbox_inches='tight')
plt.show()